# 01e -- dim_nfl_players Seed

**Purpose**: Build `dim_nfl_players` — the **central player registry** for
the fantasy league. Every player on a fantasy roster must have a row here.
Veterans are populated from nflverse at season start. Rookies are added
post-signing via ETL enrichment from `dim_rookie_prospect` (joined on `pfr_id`).

**Key**: `gsis_id` (NFL-assigned; the universal player FK for all fact tables)

**Relationship to dim_rookie_prospect**:
- Pre-signing: prospects exist only in `dim_rookie_prospect` (no `gsis_id` yet)
- Post-signing: nflverse assigns `gsis_id`; ETL merges the prospect row here
  via `pfr_id` and the `dim_rookie_prospect.gsis_id` field is back-populated

**Outputs**:
- `data/dim_nfl_players.parquet`

In [1]:
# ---- Setup & Config (shared module) ----------------------------------------
# All config + path anchoring + helpers live in notebooks/etl_helpers.py.
# CFG.data_dir / DATA / REVIEW are anchored to the repo root -> writes always
# land in <repo>/data no matter the CWD this notebook runs from.
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import (LeagueConfig, CFG, DATA, REVIEW, TODAY, ALIAS,
                         clean_player_name, generate_player_key, parse_height_to_inches)

import pandas as pd
import numpy as np
import nflreadpy as nfl
from pathlib import Path
from datetime import datetime, date
from dataclasses import dataclass, field


## 1 -- Load Raw Players

In [ ]:
# -- Load players from nflreadpy ------------------------------------------------
print("Loading NFL players from nflverse...")
try:
    players_raw = nfl.load_players().to_pandas()
except Exception as e:
    raise RuntimeError(f"Failed to load nflverse players: {e}") from e

print(f"Raw rows: {len(players_raw):,}")
print(f"Columns ({len(players_raw.columns)}): {list(players_raw.columns)}")

## 2 -- Select Columns and Save

In [ ]:
# -- Select + rename nflverse columns to canonical dim schema -----------------
# nflverse load_players() column names differ from this project's canonical
# schema. Map desired_output_col -> actual source column so values are pulled
# from the RIGHT column (the old name-only select created empty columns when the
# wished-for name was absent). Key corrections:
#   team_abbr   <- latest_team   (nflverse names current team "latest_team")
#   entry_year  <- rookie_season (season the player entered the NFL)
#   draft_club  <- draft_team
#   draft_number<- draft_pick
# Cross-ref IDs: this build provides esb_id/nfl_id/otc_id/smart_id (NOT the
# yahoo/sleeper/rotowire/etc. that the old list wished for).
_COLMAP = {
    # Identity
    "gsis_id":             "gsis_id",            # primary key (NFL GSIS)
    "display_name":        "display_name",
    "first_name":          "first_name",
    "last_name":           "last_name",
    "birth_date":          "birth_date",
    # Current roster
    "status":              "status",
    "team_abbr":           "latest_team",
    "position":            "position",
    "position_group":      "position_group",
    "jersey_number":       "jersey_number",
    "years_of_experience": "years_of_experience",
    # College / draft
    "college_name":        "college_name",
    "college_conference":  "college_conference",
    "entry_year":          "rookie_season",
    "last_season":         "last_season",
    "draft_year":          "draft_year",
    "draft_club":          "draft_team",
    "draft_round":         "draft_round",
    "draft_number":        "draft_pick",
    # Physical
    "height":              "height",
    "weight":              "weight",
    # Cross-reference IDs (this nflverse build's actual set)
    "pfr_id":              "pfr_id",
    "espn_id":             "espn_id",
    "esb_id":              "esb_id",
    "nfl_id":              "nfl_id",
    "otc_id":              "otc_id",
    "smart_id":            "smart_id",
}

_missing_src = [f"{o}<-{s}" for o, s in _COLMAP.items() if s not in players_raw.columns]
if _missing_src:
    print(f"Note: source columns absent in this nflverse build: {_missing_src}")

dim_nfl_players = pd.DataFrame({
    out_col: (players_raw[src_col] if src_col in players_raw.columns else pd.NA)
    for out_col, src_col in _COLMAP.items()
})

# Drop rows with no gsis_id (registry key)
before = len(dim_nfl_players)
dim_nfl_players = dim_nfl_players[dim_nfl_players["gsis_id"].notna()].reset_index(drop=True)
if before != len(dim_nfl_players):
    print(f"Dropped {before - len(dim_nfl_players)} rows with null gsis_id")

# Keep any id nflverse no longer returns (status DROPPED): other tables point at it
_fresh_rows = len(dim_nfl_players)
dim_nfl_players = etl.keep_dropped_ids("dim_nfl_players", dim_nfl_players)
print(f"Kept {len(dim_nfl_players) - _fresh_rows:,} ids nflverse no longer returns")

out_path = DATA / "dim_nfl_players.parquet"
etl.write_table("dim_nfl_players", dim_nfl_players)
print(f"dim_nfl_players: {len(dim_nfl_players):,} players -> {out_path}")
print(f"team_abbr populated: {dim_nfl_players['team_abbr'].notna().mean():.1%} | "
      f"entry_year populated: {dim_nfl_players['entry_year'].notna().mean():.1%}")

# Link each kept id to the id that replaced it (dim_gsis_link, #300)
_stored_links = etl.read_table(etl.GSIS_LINK_TABLE) if etl.table_exists(etl.GSIS_LINK_TABLE) else None
dim_gsis_link = etl.derive_gsis_links(dim_nfl_players, _stored_links)
etl.write_table(etl.GSIS_LINK_TABLE, dim_gsis_link)
print(f"dim_gsis_link: {len(dim_gsis_link):,} links "
      f"({len(dim_gsis_link) - (0 if _stored_links is None else len(_stored_links)):,} new)")
dim_nfl_players.head(5)

## 3 -- Validation

In [ ]:
df = etl.read_table("dim_nfl_players")
print(f"dim_nfl_players: {len(df):,} rows, {len(df.columns)} columns")
print()
print("By status:")
print(df["status"].value_counts().to_string())
print()
print("By position_group (top 10):")
print(df["position_group"].value_counts().head(10).to_string())
print()
# Cross-ref ID coverage
id_cols = ["pfr_id", "espn_id", "esb_id", "nfl_id"]
print("Cross-ref ID coverage:")
for col in id_cols:
    if col in df.columns:
        pct = df[col].notna().mean()
        print(f"  {col:<20} {pct:.1%}")